In [13]:
import torch
import gc

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

GPU memory cleared
GPU memory available: 15.83 GB
GPU memory allocated: 2.46 GB
GPU memory reserved: 15.67 GB


In [1]:
!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

import os
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re
import uuid
from rank_bm25 import BM25Okapi


INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 89.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

2025-11-16 08:08:15.090098: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763280495.321155      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763280495.375819      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [11]:
import os
import re
from typing import List, Optional
import torch
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document
from langchain import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from rank_bm25 import BM25Okapi


class LegalSearchAgent:
    # =======================================================================
    #                         INITIALIZATION
    # =======================================================================
    def __init__(self, pdf_folder: str, embeddings: HuggingFaceEmbeddings, db_path: str = "chroma_db", test_mode: bool = True):
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode
        self.embeddings = embeddings

        self.vectorstore: Optional[Chroma] = None
        self.bm25: Optional[BM25Okapi] = None
        self.bm25_docs: List[Document] = []
        self.bm25_corpus: List[List[str]] = []

        # Initialize LLM for answer generation
        self.llm = self._init_llm()

    def _init_llm(self) -> Optional[HuggingFacePipeline]:
        """Initialize LLM for answer generation"""
        try:
            print("🤖 Loading LLM for answer generation...")
            model_name = "microsoft/phi-2"
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="auto",
                trust_remote_code=True
            )
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True,
                top_p=0.95
            )
            llm = HuggingFacePipeline(pipeline=pipe)
            print("✅ LLM loaded successfully")
            return llm
        except Exception as e:
            print(f"⚠️ Could not load Phi-2: {e}")
            print("Will use extractive answers only")
            return None

    # =======================================================================
    #                         CASE NUMBER DETECTION
    # =======================================================================
    def _detect_case_number(self, query: str) -> Optional[str]:
        pattern = r"(CPLA|C\.A\.|Cr\.A|C\.P\.|HCA|RFA)[\s\-]*\d+[\s/]*(?:of\s*)?\d{4}"
        match = re.search(pattern, query, re.IGNORECASE)
        if match:
            case = match.group(0)
            case = case.replace(" ", "_").replace("of_", "_").replace("/", "_")
            case = re.sub(r"__+", "_", case)
            return case
        return None

    # =======================================================================
    #                         HYBRID RETRIEVER
    # =======================================================================
    def _retrieve_documents(self, query: str, k: int = 10) -> List[Document]:
        if self.vectorstore is None or self.bm25 is None:
            print("⚠️ Vectorstore or BM25 not built yet.")
            return []

        # Semantic search
        semantic_results = self.vectorstore.similarity_search(query, k=k)

        # BM25 keyword search
        tokens = query.split()
        bm25_scores = self.bm25.get_scores(tokens)
        bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]
        bm25_results = [self.bm25_docs[i] for i in bm25_top_idx]

        # Merge results (remove duplicates by case_number)
        combined = {}
        for doc in semantic_results + bm25_results:
            key = doc.metadata.get("case_number", doc.metadata.get("source_file", id(doc)))
            if key not in combined:
                combined[key] = doc

        return list(combined.values())[:k]

    # =======================================================================
    #                         SEARCH + ANSWER
    # =======================================================================
    def search(self, query: str, k: int = 10) -> str:
        """Search for documents and generate a structured legal answer."""
        print(f"\n🔍 QUERY: {query}")

        # Detect case number in query
        case_num = self._detect_case_number(query)

        if case_num:
            print(f"📋 Detected case number: {case_num}")
            parts = case_num.split("_")
            if len(parts) >= 2:
                case_number_only = parts[-2]
                case_year = parts[-1]

                # Retrieve more results to allow filtering
                results = self._retrieve_documents(query, k=k*3)

                # Filter to exact case match (number + year)
                exact_match = [
                    r for r in results
                    if case_number_only in r.metadata.get('case_number', '') and case_year == r.metadata.get('case_year', '')
                ]

                if exact_match:
                    results = exact_match[:k]
                    print(f"✅ Found exact case match")
                else:
                    results = results[:k]
                    print(f"⚠️ No exact case found, returning top semantic matches")
            else:
                results = self._retrieve_documents(query, k=k)
        else:
            results = self._retrieve_documents(query, k=k)

        # Print only the filenames of retrieved PDFs
        print("\n📄 Retrieved PDFs:")
        for r in results:
            print(" -", r.metadata.get("source_file", "unknown"))

        # Generate answer using all retrieved documents
        answer_text = self.generate_answer(query, results)

        # Print clean structured answer
        print("\n📌 LLM Answer:\n", answer_text)

        return answer_text

    # =======================================================================
    #                         ANSWER GENERATION
    # =======================================================================
    def generate_answer(self, query: str, documents: List[Document], use_llm: bool = True) -> str:
        """Generate a structured answer using all retrieved documents."""
        if not documents:
            return "No relevant documents found."

        # Build context from all documents
        context = "\n\n---\n\n".join([
            f"From {d.metadata.get('source_file', 'unknown')}:\n{d.page_content}"
            for d in documents
        ])

        if self.llm and use_llm:
            prompt = f"""You are a Legal Case Retrieval and Question Answering Assistant. You answer strictly using the retrieved documents provided to you. Lawyers may describe a case, ask about similar cases, ask for details of a specific case, or request information such as parties, judges, years, procedural posture, or outcomes. Your job is to provide clear, reliable, structured answers using only the supplied retrieval context.

Your response must ALWAYS follow this exact structure:

1. Final Answer:
Provide a clear, concise legal answer to the user’s query in your own words. This must be the first section. Do not mention PDFs, retrieval steps, metadata, embeddings, or your reasoning. Just answer the question directly and professionally, based only on the retrieved documents.

2. Sources Used:
List ONLY the PDF filenames that directly contributed to your answer. One per line. No commentary, no extra text.

3. Verbatim Extracts:
Copy and paste the exact sentences or paragraphs from the PDFs that support your answer. These must be word-for-word quotes. Under each quote, clearly mention the PDF filename it comes from.

Rules:
- You must not hallucinate any information or case details that are not present in the retrieved documents.
- If the answer cannot be found in the retrieved documents, respond in the Final Answer section: "The retrieved documents do not contain the required information."
- Use only the content in the retrieved documents. Never add external legal knowledge.
"""

            # Append retrieved documents to prompt
            prompt += "\n\nRETRIEVED DOCUMENTS:\n" + context

            try:
                result = self.llm.invoke(prompt)
                if isinstance(result, list) and len(result) > 0:
                    answer_text = result[0].get("generated_text", str(result))
                elif isinstance(result, str):
                    answer_text = result
                else:
                    answer_text = str(result)
                return answer_text.strip()
            except Exception as e:
                print(f"⚠️ LLM error: {e}, falling back to extractive answer")
                return self._extractive_answer(documents)
        else:
            return self._extractive_answer(documents)

    def _extractive_answer(self, documents: List[Document]) -> str:
        """Fallback extractive answer if LLM fails."""
        answer = "Based on the retrieved documents:\n\n"
        for i, doc in enumerate(documents[:3], 1):
            answer += f"{i}. From {doc.metadata.get('source_file', 'unknown')}:\n"
            answer += f"   {doc.page_content[:300]}...\n\n"
        return answer


In [12]:
import shutil
import os
import torch
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import Document
from rank_bm25 import BM25Okapi

print("=" * 70)
print("COPYING DB TO WRITABLE LOCATION")
print("=" * 70)

# Copy from read-only input to writable workspace
src = "/kaggle/input/fyp-vector-store/chroma_db"
dst = "/kaggle/working/chroma_db"

if os.path.exists(dst):
    shutil.rmtree(dst)

print(f"\nCopying from: {src}")
print(f"Copying to: {dst}")
shutil.copytree(src, dst)
print("✅ Copied successfully")

# Now load from writable location
print("\n📚 Loading embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda'}
)

print("\n🔧 Initializing agent...")
agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",
    embeddings=embeddings,
    db_path="/kaggle/working/chroma_db",  # ← Use writable location
    test_mode=False
)

print("\n📦 Loading vector store...")
agent.vectorstore = Chroma(
    persist_directory="/kaggle/working/chroma_db",
    embedding_function=embeddings
)
chunk_count = agent.vectorstore._collection.count()
print(f"✅ Loaded {chunk_count} chunks")

print("\n🔨 Rebuilding BM25...")
vectorstore_data = agent.vectorstore.get()

docs = []
for i, content in enumerate(vectorstore_data['documents']):
    metadata = vectorstore_data['metadatas'][i]
    doc = Document(page_content=content, metadata=metadata)
    docs.append(doc)

agent.bm25_docs = docs
agent.bm25_corpus = [doc.page_content.split() for doc in docs]
agent.bm25 = BM25Okapi(agent.bm25_corpus)
print(f"✅ BM25 ready with {len(agent.bm25_docs)} documents")

print("\n" + "=" * 70)
print("✅ READY TO USE")
print("=" * 70)

# Test


COPYING DB TO WRITABLE LOCATION

Copying from: /kaggle/input/fyp-vector-store/chroma_db
Copying to: /kaggle/working/chroma_db
✅ Copied successfully

📚 Loading embeddings...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 20.12 MiB is free. Process 2737 has 14.72 GiB memory in use. Of the allocated memory 14.56 GiB is allocated by PyTorch, and 34.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [15]:
query = "What was CPLA 210 of 2024 about?"
answer = agent.search(query, k=5)
print(answer)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔍 QUERY: What was CPLA 210 of 2024 about?
📋 Detected case number: CPLA_210_2024
✅ Found exact case match

📄 Retrieved PDFs:
 - C.P.L.A.210_2024.pdf

📌 LLM Answer:
 You are a Legal Case Retrieval and Question Answering Assistant. You answer strictly using the retrieved documents provided to you. Lawyers may describe a case, ask about similar cases, ask for details of a specific case, or request information such as parties, judges, years, procedural posture, or outcomes. Your job is to provide clear, reliable, structured answers using only the supplied retrieval context.

Your response must ALWAYS follow this exact structure:

1. Final Answer:
Provide a clear, concise legal answer to the user’s query in your own words. This must be the first section. Do not mention PDFs, retrieval steps, metadata, embeddings, or your reasoning. Just answer the question directly and professionally, based only on the retrieved documents.

2. Sources Used:
List ONLY the PDF filenames that directly contribu

In [17]:
# ============================================================================
# TEST CASES - Different query types
# ============================================================================

test_cases = [
    # 1. Exact case number
    {
        "type": "EXACT CASE NUMBER",
        "query": "What was CA 570 of 2011 about?",
        "expected_case": "C.A.570_2011",
        "expected_topic": "Customs Act amendment discrimination"
    },
    
    # 2. Case by year + topic
    {
        "type": "YEAR + TOPIC",
        "query": "Supreme Court case from 2015 about inheritance and succession rights",
        "expected_case": "C.A.1002_2015",
        "expected_topic": "Inheritance mutation Sunni Shia"
    },
    
    # 3. Recent tax case
    {
        "type": "RECENT CASE + SPECIFIC ISSUE",
        "query": "What is the Supreme Court ruling on definite information in income tax 2024?",
        "expected_case": "C.P.L.A.862_2024",
        "expected_topic": "definite information Income Tax amendment assessment"
    },
    
    # 4. Judge-based search
    {
        "type": "JUDGE NAME",
        "query": "Cases decided by Justice Yahya Afridi related to taxation",
        "expected_case": "C.A.570_2011 or C.A.1002_2015 or C.P.L.A.862_2024",
        "expected_topic": "Multiple cases by Yahya Afridi"
    },
    
    # 5. Party-based search
    {
        "type": "PARTY SEARCH",
        "query": "Federation of Pakistan vs Saleem Raza customs case",
        "expected_case": "C.A.570_2011",
        "expected_topic": "Federation FBR Saleem Raza"
    },
    
    # 6. Legal principle search
    {
        "type": "LEGAL PRINCIPLE",
        "query": "What is the doctrine of judicial deference in Pakistani law?",
        "expected_case": "C.A.570_2011",
        "expected_topic": "judicial deference constitutional court fiscal statute"
    },
    
    # 7. Sectarian law (inheritance)
    {
        "type": "SECTARIAN LAW",
        "query": "How does Sunni and Shia Islamic law differ in inheritance?",
        "expected_case": "C.A.1002_2015",
        "expected_topic": "Hanafi Sunni Shia inheritance law"
    },
    
    # 8. Limitation period search
    {
        "type": "PROCEDURAL/LIMITATION",
        "query": "Six year limitation period for suit declaration",
        "expected_case": "C.A.1002_2015",
        "expected_topic": "Limitation Act Article 120"
    },
]

# ============================================================================
# RUN TESTS
# ============================================================================

test_results = []

for i, test in enumerate(test_cases, 1):
    print(f"\n{'='*80}")
    print(f"TEST {i}: {test['type']}")
    print(f"{'='*80}")
    print(f"Query: {test['query']}")
    print(f"Expected Case: {test['expected_case']}")
    print(f"Expected Topic: {test['expected_topic']}\n")
    
    # Search
    docs = agent.search(test['query'], k=5)
    
    # Check if expected case is in results
    retrieved_cases = [d.metadata.get('case_number') for d in docs]
    expected_cases = test['expected_case'].split(' or ')
    
    found = any(exp in ' '.join(retrieved_cases) for exp in expected_cases)
    
    print(f"Retrieved cases (top 5):")
    for j, doc in enumerate(docs, 1):
        print(f"  {j}. {doc.metadata['case_number']} ({doc.metadata['source_file']})")
    
    # Generate answer
    if agent.llm:
        print(f"\nGenerating answer...")
        answer = agent.generate_answer(test['query'], docs)
        print(f"Answer: {answer}...")
    else:
        print(f"\nLLM not available, using extractive answer")
        answer = agent._extractive_answer(docs)
        print(f"Answer: {answer[:400]}...")
    
    # Record result
    result = {
        "test_num": i,
        "type": test['type'],
        "query": test['query'],
        "expected": test['expected_case'],
        "retrieved": retrieved_cases,
        "found": found,
        "status": "✅ PASS" if found else "❌ FAIL"
    }
    test_results.append(result)
    
    print(f"\nStatus: {result['status']}")

# ============================================================================
# SUMMARY
# ============================================================================

print(f"\n\n{'='*80}")
print("TEST SUMMARY")
print(f"{'='*80}\n")

passed = sum(1 for r in test_results if r['found'])
total = len(test_results)

print(f"Tests Passed: {passed}/{total} ({100*passed/total:.1f}%)\n")

for result in test_results:
    print(f"{result['status']} | {result['type']:30s} | Retrieved: {result['retrieved'][0]}")

print(f"\n{'='*80}")
print("KEY FINDINGS:")
print(f"{'='*80}")
print(f"""
1. Exact case number matching: {'✅' if any(r['type'] == 'EXACT CASE NUMBER' and r['found'] for r in test_results) else '❌'}
2. Topic + year matching: {'✅' if any(r['type'] == 'YEAR + TOPIC' and r['found'] for r in test_results) else '❌'}
3. Semantic search (judge, party, principles): {'✅' if any(r['type'] in ['JUDGE NAME', 'PARTY SEARCH', 'LEGAL PRINCIPLE'] and r['found'] for r in test_results) else '❌'}
4. Legal concept search: {'✅' if any(r['type'] == 'SECTARIAN LAW' and r['found'] for r in test_results) else '❌'}
5. Procedural/technical search: {'✅' if any(r['type'] == 'PROCEDURAL/LIMITATION' and r['found'] for r in test_results) else '❌'}
""")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



TEST 1: EXACT CASE NUMBER
Query: What was CA 570 of 2011 about?
Expected Case: C.A.570_2011
Expected Topic: Customs Act amendment discrimination


🔍 QUERY: What was CA 570 of 2011 about?

📄 Retrieved PDFs:
 - C.A.570_2011.pdf
 - C.A.722_2012.pdf
 - C.A.836-L_2013.pdf
 - C.A.1521_2018.pdf
 - C.A.649_2019.pdf
Retrieved cases (top 5):
  1. C.A.570_2011 (C.A.570_2011.pdf)
  2. C.A.722_2012 (C.A.722_2012.pdf)
  3. C.A.836-L_2013 (C.A.836-L_2013.pdf)
  4. C.A.1521_2018 (C.A.1521_2018.pdf)
  5. C.A.649_2019 (C.A.649_2019.pdf)

Generating answer...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer: You are a legal research assistant. Based ONLY on the provided court judgment excerpts, answer the question concisely and accurately. Do not make up information. Do not generate follow-up questions or exercises.

Question: What was CA 570 of 2011 about?

Court Judgment Excerpts:
From C.A.570_2011.pdf:
[Case Type: C.A.570] [Year: 2011] [Case Number: C.A.570_2011]

legislature, and where two diverse views are reasonably possible, 
                                       
1      I.A Sherwani’s case (1991 SCMR 1041) 
2      Budget Instructions for the year 2010-2011 
vide letter  No. 6(1)/2010-CB    dated 5-6-2010

---

From C.A.722_2012.pdf:
[Case Type: C.A.722] [Year: 2012] [Case Number: C.A.722_2012]

CA-722/2012, etc  12 
 
Scope of Section 1(3) as to the retrospective effect of the 2011 Act on 
arbitration agreements  
22. As for subsection (3) of Section 1 of the 2011 Act, which states that 
the Act shall apply to arbitration agreements made before the date of 
commencement of

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

Answer: You are a legal research assistant. Based ONLY on the provided court judgment excerpts, answer the question concisely and accurately. Do not make up information. Do not generate follow-up questions or exercises.

Question: Federation of Pakistan vs Saleem Raza customs case

Court Judgment Excerpts:
From C.P.L.A.78-K_2024.pdf:
[Case Type: C.P.L.A.78_K] [Year: 2024] [Case Number: C.P.L.A.78-K_2024]

extinguished once the imported goods have crossed the customs barrier. 
The Federal Board of Revenue (the “ FBR”), acting through its 
Collectorates and the Directorate of Post Clearance Audit (the “ PCA”), 
now seeks leave to appeal against those decisions. 
 
3. For the purpose of completeness, it would be pertinent to note that 
the learned Single Bench of the Lahore High Court vide judgment dated 
30.11.20181 held otherwise , and declared that the authority of the 
Customs to assess and recover sales tax and advance income tax 
extended beyond the clearance of the goods. We have b

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer: You are a legal research assistant. Based ONLY on the provided court judgment excerpts, answer the question concisely and accurately. Do not make up information. Do not generate follow-up questions or exercises.

Question: What is the doctrine of judicial deference in Pakistani law?

Court Judgment Excerpts:
From C.A.749_2013.pdf:
[Case Type: C.A.749] [Year: 2013] [Case Number: C.A.749_2013]

CA No. 749/2013, etc. 6
Unlike other institutions and all those who are in the service of 
Pakistan, Judges are accountable  to themselves , therefore, they 
must be beyond reproach in their work ethic. The credibility of an 
institution on which a substantial amount is spent from  the public 
exchequer and in which hundreds are employed, must not be 
allowed to be undermined. We must assiduously and diligently 
strive to decide cases.  
 
15. An indispensable component of dispensing justice is to 
deliver judgments within a reasonable ti me. ‘To no one will we 
refuse or delay, right or j

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer: You are a legal research assistant. Based ONLY on the provided court judgment excerpts, answer the question concisely and accurately. Do not make up information. Do not generate follow-up questions or exercises.

Question: How does Sunni and Shia Islamic law differ in inheritance?

Court Judgment Excerpts:
From C.A.1002_2015.pdf:
[Case Type: C.A.1002] [Year: 2015] [Case Number: C.A.1002_2015]

remanded by the Collector for afresh decision that  she, for the first time, 
took the stance that  her son, Taj Muhammad, belonged to  Shia sect. 
Moreover, we also note d that, she was not a cre dible witness , as her 
deposition that her husband , namely Noor Muhammad alias Nooran, 
father of Taj Muhammad, was a Shia Muslim was belied by th e 
inheritance mutation of Noor Muhammad alias Nooran (Exh. P 4). Under 
the said mutation, the estate of Noor  Muhammad alias Nooran was 
divided amongst his legal heirs in accordance with the Hanfi Sunni law of 
inheritance, not Shia law.  
14. Mo

In [18]:

print("=" * 80)
print("LLM PROMPT OPTIMIZATION TEST")
print("=" * 80)

# Initialize and load
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",
    embeddings=embeddings,
    db_path="/kaggle/working/chroma_db",
    test_mode=False
)

agent.vectorstore = Chroma(
    persist_directory="/kaggle/working/chroma_db",
    embedding_function=embeddings
)

vectorstore_data = agent.vectorstore.get()
docs = [Document(page_content=c, metadata=m) for c, m in zip(vectorstore_data['documents'], vectorstore_data['metadatas'])]
agent.bm25_docs = docs
agent.bm25_corpus = [doc.page_content.split() for doc in docs]
agent.bm25 = BM25Okapi(agent.bm25_corpus)

print(f"✅ System loaded with {len(agent.bm25_docs)} documents\n")

# ============================================================================
# SINGLE TEST QUESTION
# ============================================================================

query = "What was CA 570 of 2011 about?"
print(f"TEST QUESTION: {query}\n")

# Retrieve documents once
retrieved_docs = agent.search(query, k=5)
sources = " | ".join([d.metadata.get('source_file', 'unknown') for d in retrieved_docs[]])
context = "\n\n---\n\n".join([
    f"{d.metadata.get('source_file', 'unknown')}:\n{d.page_content}"
    for d in retrieved_docs[]
])

print(f"Retrieved documents: {sources}\n")
print("=" * 80)

# ============================================================================
# DIFFERENT PROMPTS TO TEST
# ============================================================================

prompts = [
    {
        "name": "PROMPT 1: Very Strict (2-3 sentences only)",
        "prompt": f"""Answer in 2-3 sentences ONLY. Nothing else.

Q: {query}
Context: {context}

A:"""
    },
    
    {
        "name": "PROMPT 2: Legal Brief Format",
        "prompt": f"""You are a legal research assistant. Provide a brief answer (2-3 sentences).

Question: {query}

Case Information:
{context}

Brief Answer:"""
    },
    
    {
        "name": "PROMPT 3: Summarize in Plain Language",
        "prompt": f"""Explain this case simply in 2-3 sentences for someone not a lawyer.

Case: {query}

Details:
{context}

Summary:"""
    },
    
    {
        "name": "PROMPT 4: Facts-Issue-Decision Format",
        "prompt": f"""Answer with: (1) What happened, (2) What was the legal issue, (3) How did the court rule.

Question: {query}

Source:
{context}

Answer:"""
    },
    
    {
        "name": "PROMPT 5: Direct Answer Only",
        "prompt": f"""{query}

Context:
{context}

Answer (one paragraph):"""
    },
    
    {
        "name": "PROMPT 6: JSON Format",
        "prompt": f"""Extract the case information in JSON format. Include: case_number, parties, issue, decision.

Case: {query}

Context:
{context}

JSON:"""
    },
    
    {
        "name": "PROMPT 7: Citation Format (Academic)",
        "prompt": f"""Describe the case with proper legal citations.

Question: {query}

Text:
{context}

Description:"""
    },
    
    {
        "name": "PROMPT 8: Role-Play as Legal AI",
        "prompt": f"""You are a legal research AI. A lawyer asks: "{query}"

Your answer (2 sentences maximum):

{context}

Answer:"""
    },
    
    {
        "name": "PROMPT 9: Bullet Points",
        "prompt": f"""Answer the question using bullet points:

Q: {query}

Context:
{context}

Answer:
•""",
    },
    
    {
        "name": "PROMPT 10: Temperature/Style Test",
        "prompt": f"""Answer directly without explanation.

Q: {query}

Answer:
{context}""",
    },
]

# ============================================================================
# TEST EACH PROMPT
# ============================================================================

results = []

for i, prompt_config in enumerate(prompts, 1):
    print(f"\n{'='*80}")
    print(f"TEST {i}: {prompt_config['name']}")
    print(f"{'='*80}\n")
    
    print("PROMPT:")
    print("-" * 80)
    print(prompt_config['prompt'][:300] + "..." if len(prompt_config['prompt']) > 300 else prompt_config['prompt'])
    print("-" * 80 + "\n")
    
    try:
        result = agent.llm.invoke(prompt_config['prompt'])
        
        if isinstance(result, list) and len(result) > 0:
            answer_text = result[0].get("generated_text", str(result))
        elif isinstance(result, str):
            answer_text = result
        else:
            answer_text = str(result)
        
        # Clean up
        if prompt_config['name'].split(':')[1].strip() == "JSON Format":
            # For JSON, extract just the JSON
            if "{" in answer_text:
                answer_text = answer_text[answer_text.index("{"):]
                if "}" in answer_text:
                    answer_text = answer_text[:answer_text.rindex("}")+1]
        else:
            # Remove prompt repetition
            for marker in ["Q:", "Question:", "Answer:", "A:", "Answer (", "Summary:"]:
                if marker in answer_text:
                    answer_text = answer_text.split(marker)[-1].strip()
            
            # Remove garbage
            for garbage in ["Exercise:", "Follow-up", "Solution:", "1.", "2.", "•", "Note:","Based on"]:
                if garbage in answer_text:
                    answer_text = answer_text.split(garbage)[0].strip()
        
        answer_text = answer_text.strip()
        
        # Count tokens in response
        token_count = len(answer_text.split())
        
        print("OUTPUT:")
        print("-" * 80)
        print(answer_text[:400])
        if len(answer_text) > 400:
            print(f"...[truncated, {token_count} words total]")
        print("-" * 80)
        
        # Score the output
        is_concise = token_count < 100
        has_no_garbage = not any(g in answer_text for g in ["Exercise:", "Follow-up", "Question:", "Solution:"])
        is_relevant = any(keyword in answer_text.lower() for keyword in ["customs", "570", "amendment", "discrimination", "pakistan"])
        
        score = sum([is_concise, has_no_garbage, is_relevant])
        
        print(f"\nQUALITY METRICS:")
        print(f"  Concise (<100 words): {'✅' if is_concise else '❌'} ({token_count} words)")
        print(f"  No Garbage Output: {'✅' if has_no_garbage else '❌'}")
        print(f"  Relevant Content: {'✅' if is_relevant else '❌'}")
        print(f"  Overall Score: {score}/3")
        
        results.append({
            "prompt_num": i,
            "name": prompt_config['name'],
            "score": score,
            "word_count": token_count,
            "concise": is_concise,
            "no_garbage": has_no_garbage,
            "relevant": is_relevant,
            "output": answer_text[:200]
        })
        
    except Exception as e:
        print(f"❌ ERROR: {str(e)[:200]}")
        results.append({
            "prompt_num": i,
            "name": prompt_config['name'],
            "score": 0,
            "error": str(e)[:100]
        })

# ============================================================================
# SUMMARY AND RANKING
# ============================================================================

print(f"\n\n{'='*80}")
print("SUMMARY AND RANKING")
print(f"{'='*80}\n")

# Sort by score
sorted_results = sorted([r for r in results if 'score' in r], key=lambda x: x['score'], reverse=True)

print("RANKED PROMPTS (by quality):\n")
for rank, result in enumerate(sorted_results, 1):
    print(f"{rank}. {result['name']}")
    print(f"   Score: {result['score']}/3 | Words: {result['word_count']} | Output: {result['output'][:80]}...")
    print()

print(f"\n{'='*80}")
print("BEST PROMPT")
print(f"{'='*80}")
best = sorted_results[0]
print(f"\n✅ WINNER: {best['name']}")
print(f"   Score: {best['score']}/3")
print(f"   Word Count: {best['word_count']}")
print(f"\nRecommended prompt structure:")
print(f"   - Use simple, direct instructions")
print(f"   - Limit output length (2-3 sentences)")
print(f"   - Avoid asking for follow-ups or exercises")
print(f"   - Be explicit about what NOT to include")

LLM PROMPT OPTIMIZATION TEST
🤖 Loading LLM for answer generation...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


✅ LLM loaded successfully


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

OUTPUT:
--------------------------------------------------------------------------------
What was CA 570 of 2011 about?

Text:
C.A.570_201
--------------------------------------------------------------------------------

QUALITY METRICS:
  Concise (<100 words): ✅ (9 words)
  No Garbage Output: ✅
  Relevant Content: ✅
  Overall Score: 3/3

TEST 8: PROMPT 8: Role-Play as Legal AI

PROMPT:
--------------------------------------------------------------------------------
You are a legal research AI. A lawyer asks: "What was CA 570 of 2011 about?"

Your answer (2 sentences maximum):

C.A.570_2011.pdf:
[Case Type: C.A.570] [Year: 2011] [Case Number: C.A.570_2011]

legislature, and where two diverse views are reasonably possible, 
                                      ...
--------------------------------------------------------------------------------



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


OUTPUT:
--------------------------------------------------------------------------------
C.A.570_201
--------------------------------------------------------------------------------

QUALITY METRICS:
  Concise (<100 words): ✅ (1 words)
  No Garbage Output: ✅
  Relevant Content: ✅
  Overall Score: 3/3

TEST 9: PROMPT 9: Bullet Points

PROMPT:
--------------------------------------------------------------------------------
Answer the question using bullet points:

Q: What was CA 570 of 2011 about?

Context:
C.A.570_2011.pdf:
[Case Type: C.A.570] [Year: 2011] [Case Number: C.A.570_2011]

legislature, and where two diverse views are reasonably possible, 
                                       
1      I.A Sherwani’s case...
--------------------------------------------------------------------------------



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


OUTPUT:
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------

QUALITY METRICS:
  Concise (<100 words): ✅ (0 words)
  No Garbage Output: ✅
  Relevant Content: ❌
  Overall Score: 2/3

TEST 10: PROMPT 10: Temperature/Style Test

PROMPT:
--------------------------------------------------------------------------------
Answer directly without explanation.

Q: What was CA 570 of 2011 about?

Answer:
C.A.570_2011.pdf:
[Case Type: C.A.570] [Year: 2011] [Case Number: C.A.570_2011]

legislature, and where two diverse views are reasonably possible, 
                                       
1      I.A Sherwani’s case (199...
--------------------------------------------------------------------------------

OUTPUT:
--------------------------------------------------------------------------------
C.A.570_201
--------------------------------------------------------------------------------

QUALIT